# Lesson 6: Linear Regression

*ML Academy — companion notebook*

Fit a line (and a plane) to data, derive the least-squares solution, and interpret the weights.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-06.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. In Python: from scratch, then with scikit-learn

In [1]:
import numpy as np

x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 4, 5, 4, 5], dtype=float)

# Simple formulas
w = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
b = y.mean() - w * x.mean()
print(w, b)                         # 0.6 2.2

# Normal equation with a column of ones for the bias
X = np.column_stack([np.ones_like(x), x])
w_vec = np.linalg.solve(X.T @ X, X.T @ y)   # solves (XᵀX) w = Xᵀy
print(w_vec)                        # [2.2 0.6]

y_hat = X @ w_vec
r2 = 1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2)
print(round(r2, 3))                 # 0.6

0.6 2.2
[2.2 0.6]
0.6


In [2]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)
print(model.coef_.round(1))          # one weight per feature
print(round(model.intercept_, 1))
print(round(r2_score(y_test, model.predict(X_test)), 3))   # R² on unseen data

[  37.9 -242.   542.4  347.7 -931.5  518.1  163.4  275.3  736.2   48.7]
151.3
0.453


## 14. Going deeper: the statistics behind linear regression

In [3]:
import numpy as np
from scipy import stats
from sklearn.datasets import load_diabetes

def ols(X, y):
    """Least squares plus standard errors, t, p-values and 95% intervals."""
    n, d = X.shape
    A = np.column_stack([np.ones(n), X])            # add the bias column
    w = np.linalg.solve(A.T @ A, A.T @ y)           # the usual weights
    resid = y - A @ w
    dof = n - d - 1                                 # data points minus parameters
    sigma2 = resid @ resid / dof                    # estimated noise variance
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(A.T @ A)))
    t = w / se                                      # how many SEs away from zero
    p = 2 * stats.t.sf(np.abs(t), dof)              # two-sided p-values
    tc = stats.t.ppf(0.975, dof)                    # 95% critical value
    return w, se, t, p, w - tc * se, w + tc * se

data = load_diabetes()
X, y, names = data.data, data.target, data.feature_names
w, se, t, p, lo, hi = ols(X, y)
print(f"{'':6}{'weight':>8}{'SE':>7}{'t':>7}{'p':>7}   95% interval")
for row in zip(["bias"] + names, w, se, t, p, lo, hi):
    print("{:6}{:8.1f}{:7.1f}{:7.2f}{:7.3f}   [{:7.1f}, {:7.1f}]".format(*row))

        weight     SE      t      p   95% interval
bias     152.1    2.6  59.06  0.000   [  147.1,   157.2]
age      -10.0   59.7  -0.17  0.867   [ -127.4,   107.4]
sex     -239.8   61.2  -3.92  0.000   [ -360.1,  -119.5]
bmi      519.8   66.5   7.81  0.000   [  389.1,   650.6]
bp       324.4   65.4   4.96  0.000   [  195.8,   453.0]
s1      -792.2  416.7  -1.90  0.058   [-1611.2,    26.8]
s2       476.7  339.0   1.41  0.160   [ -189.6,  1143.1]
s3       101.0  212.5   0.48  0.635   [ -316.7,   518.8]
s4       177.1  161.5   1.10  0.273   [ -140.3,   494.4]
s5       751.3  171.9   4.37  0.000   [  413.4,  1089.1]
s6        67.6   66.0   1.02  0.306   [  -62.1,   197.3]


In [4]:
from sklearn.linear_model import LinearRegression

# VIF of feature j = 1 / (1 - R²) when feature j is predicted from all the other features
for j, name in enumerate(names):
    others = np.delete(X, j, axis=1)
    r2_j = LinearRegression().fit(others, X[:, j]).score(others, X[:, j])
    print(f"{name:4} VIF {1 / (1 - r2_j):5.1f}")

# Drop s2 and refit: s1's standard error shrinks, the fit barely changes
keep = [j for j, name in enumerate(names) if name != "s2"]
w2, se2, *_ = ols(X[:, keep], y)
print("s1 with s2:    weight", round(w[5], 1), " SE", round(se[5], 1))
print("s1 without s2: weight", round(w2[5], 1), " SE", round(se2[5], 1))
print("R² with s2:   ", round(LinearRegression().fit(X, y).score(X, y), 4))
print("R² without s2:", round(LinearRegression().fit(X[:, keep], y).score(X[:, keep], y), 4))

age  VIF   1.2
sex  VIF   1.3
bmi  VIF   1.5
bp   VIF   1.5
s1   VIF  59.2
s2   VIF  39.2
s3   VIF  15.4
s4   VIF   8.9
s5   VIF  10.1
s6   VIF   1.5
s1 with s2:    weight -792.2  SE 416.7
s1 without s2: weight -228.5  SE 114.0
R² with s2:    0.5177
R² without s2: 0.5155


In [5]:
from sklearn.model_selection import cross_val_score

n = len(y)
rng = np.random.default_rng(0)
print("noise cols  train R²  adjusted R²  5-fold CV R²")
for extra in [0, 10, 50, 100]:
    Xe = np.column_stack([X, rng.normal(size=(n, extra))])     # add columns of pure noise
    d = Xe.shape[1]
    r2 = LinearRegression().fit(Xe, y).score(Xe, y)
    adj = 1 - (1 - r2) * (n - 1) / (n - d - 1)
    cv = cross_val_score(LinearRegression(), Xe, y, cv=5, scoring="r2").mean()
    print(f"{extra:10d}  {r2:8.3f}  {adj:11.3f}  {cv:12.3f}")

noise cols  train R²  adjusted R²  5-fold CV R²
         0     0.518        0.507         0.482
        10     0.530        0.508         0.469
        50     0.582        0.516         0.398
       100     0.633        0.511         0.277


In [6]:
# Salaries whose spread grows with experience (simulated, in lakh per year)
rng = np.random.default_rng(2)
years = rng.uniform(0, 30, size=300)
salary = 4 * np.exp(0.06 * years) * rng.lognormal(0, 0.25, size=300)

def spread(x, target):
    """Fit a line, then compare residual spread for the lowest and highest third of x."""
    m = LinearRegression().fit(x[:, None], target)
    r = target - m.predict(x[:, None])
    low, high = x < np.quantile(x, 1/3), x > np.quantile(x, 2/3)
    return m, r[low].std(), r[high].std()

_, a, b = spread(years, salary)
print(f"raw salary: spread {a:.2f} (junior) vs {b:.2f} (senior)")
m_log, a, b = spread(years, np.log(salary))
print(f"log salary: spread {a:.3f} (junior) vs {b:.3f} (senior)")
k = m_log.coef_[0]
print(f"weight {k:.4f} -> each extra year multiplies salary by {np.exp(k):.3f}")

raw salary: spread 1.53 (junior) vs 5.20 (senior)
log salary: spread 0.220 (junior) vs 0.249 (senior)
weight 0.0591 -> each extra year multiplies salary by 1.061


In [7]:
from sklearn.linear_model import HuberRegressor

rng = np.random.default_rng(1)
x = np.arange(1, 21, dtype=float)
y_ = 2 * x + 5 + rng.normal(0, 2, size=20)         # true line: y = 2x + 5
x = np.append(x, 40.0); y_ = np.append(y_, 20.0)   # one unusual point far to the right

A = np.column_stack([np.ones_like(x), x])
lev = np.diag(A @ np.linalg.inv(A.T @ A) @ A.T)    # leverage = diagonal of the hat matrix
w_ = np.linalg.solve(A.T @ A, A.T @ y_)
e = y_ - A @ w_
p_ = A.shape[1]
s2 = e @ e / (len(y_) - p_)
cooks = e**2 / (p_ * s2) * lev / (1 - lev) ** 2     # Cook's distance

print("slope, least squares with the point:   ", round(w_[1], 2))
print("slope, least squares without the point:", round(LinearRegression().fit(x[:-1, None], y_[:-1]).coef_[0], 2))
print("slope, Huber with the point:           ", round(HuberRegressor().fit(x[:, None], y_).coef_[0], 2))
print("leverage: median", round(np.median(lev), 3), "| unusual point", round(lev[-1], 3))
print("Cook's D: largest other", round(cooks[:-1].max(), 3), "| unusual point", round(cooks[-1], 2))

slope, least squares with the point:    0.75
slope, least squares without the point: 1.93
slope, Huber with the point:            1.85
leverage: median 0.065 | unusual point 0.576
Cook's D: largest other 0.099 | unusual point 12.73


In [8]:
import pandas as pd

df = pd.read_csv("../data/houses.csv")
all3 = pd.get_dummies(df["area"], dtype=float)           # Central, Outskirts, Suburb
A = np.column_stack([np.ones(len(df)), df["size_m2"], all3])
print("columns:", A.shape[1], " independent columns (rank):", np.linalg.matrix_rank(A))

Xd = pd.get_dummies(df[["size_m2", "area"]], columns=["area"], drop_first=True, dtype=float)
m = LinearRegression().fit(Xd, df["price_lakh"])
for name, wt in zip(Xd.columns, m.coef_):
    print(f"{name:15}{wt:7.2f}")
print(f"{'bias':15}{m.intercept_:7.2f}")

columns: 5  independent columns (rank): 4


size_m2           0.61
area_Outskirts  -41.13
area_Suburb     -14.79
bias             22.98


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.linear_model import LinearRegression
m1 = LinearRegression().fit(df[["size_m2"]], df["price_lakh"])
print("price = %.3f * size + %.2f" % (m1.coef_[0], m1.intercept_), " R2 =", round(m1.score(df[["size_m2"]], df["price_lakh"]), 3))
cols = ["size_m2", "dist_km", "age_yrs"]
m3 = LinearRegression().fit(df[cols], df["price_lakh"])
print({c: round(float(v), 3) for c, v in zip(cols, m3.coef_)}, round(m3.intercept_, 2), " R2 =", round(m3.score(df[cols], df["price_lakh"]), 3))

price = 0.290 * size + 41.11  R2 = 0.187
{'size_m2': 0.715, 'dist_km': -1.985, 'age_yrs': -0.988} 29.0  R2 = 0.913


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: The least-squares slope

Compute the slope with the formula from the lesson.

In [11]:
import numpy as np
x = np.array([1, 2, 3, 4, 5.0]); y = np.array([2, 4, 5, 4, 5.0])
xm, ym = x.mean(), y.mean()
w = None  # ← your code here
check(w, 0.6, hint='w = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)². It should be 0.6.')

❌ Not yet. Hint: w = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)². It should be 0.6.


### Challenge 2: R²

Compute R² for the predictions of the line ŷ = 0.6x + 2.2.

In [12]:
import numpy as np
x = np.array([1, 2, 3, 4, 5.0]); y = np.array([2, 4, 5, 4, 5.0])
y_hat = 0.6 * x + 2.2
ss_res = np.sum((y - y_hat) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = None  # ← your code here
check(r2, 0.6000000000000001, hint='1 − SS_res / SS_tot.')

❌ Not yet. Hint: 1 − SS_res / SS_tot.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. The least-squares slope** — the missing line:

```python
w = np.sum((x - xm) * (y - ym)) / np.sum((x - xm) ** 2)
```

**2. R²** — the missing line:

```python
r2 = 1 - ss_res / ss_tot
```